Fine Tune Dialo-GPT

In [ ]:
print("konfigurasi training...")
training_args = TrainingArguments(
    output_dir="/kaggle/working/chatbot_fitness_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=8,   # 8 VRAM GPU T4 tidak jebol (OOM)
    save_steps=500,                  # Simpan progress setiap 500 langkah
    save_total_limit=2,
    fp16=False,
    report_to="none"                 # Mematikan notifikasi Weights & Biases (wandb) agar tidak ribet login
)

# Inisialisasi Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Memulai proses Fine-Tuning! Silakan tunggu... 🚀")
trainer.train()

# Simpan Model dan Tokenizer final ke folder Kaggle
trainer.save_model("/kaggle/working/chatbot_fitness_final")
tokenizer.save_pretrained("/kaggle/working/chatbot_fitness_final")

print("Selesai! Model chatbot fitness-mu berhasil disimpan di folder /kaggle/working/chatbot_fitness_final")

Fine Tune LoRA

In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# 2. Load Dataset Hasil Cleaning (10.000 baris)
print("Memuat dataset...")
df = pd.read_csv('/kaggle/working/dataset_fitness_bersih_10k.csv')
dataset = Dataset.from_pandas(df)

# 3. Format Prompt ke Standar ChatML (Spesifik untuk Qwen)
def format_chatml(example):
    # Membungkus Question dan Answer dengan tag khusus Qwen
    text = f"<|im_start|>user\n{example['question']}<|im_end|>\n<|im_start|>assistant\n{example['answer']}<|im_end|>"
    return {"text": text}

print("Menerapkan format ChatML...")
dataset = dataset.map(format_chatml)

# 4. Panggil Tokenizer & Model Qwen2.5 (1.5 Miliar Parameter)
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Menggunakan eos_token sebagai pad_token agar tidak error saat batching
tokenizer.pad_token = tokenizer.eos_token

print("Mengunduh otak Qwen2.5...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map={"": 0},          # Fitur ajaib agar Kaggle otomatis membagi beban ke 2 GPU T4!
    dtype=torch.float16   # Wajib pakai presisi 16-bit agar hemat VRAM
)

# 5. Pasang Adaptor LoRA (Kunci utama agar GPU tidak OOM)
peft_config = LoraConfig(
    r=8,                                   # Rank matriks LoRA (semakin kecil semakin hemat memori)
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],   # Bagian "Attention" yang dipasangi adaptor
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Tempelkan adaptor ke model asli
model = get_peft_model(model, peft_config)
print("\nStatistik Parameter yang Dilatih (Hanya adaptornya saja):")
model.print_trainable_parameters()

# 6. Konfigurasi Pelatihan (Training Arguments)
training_args = SFTConfig(
    output_dir="/kaggle/working/qwen_fitness_checkpoints",
    num_train_epochs=3,                  # 3 putaran sudah cukup
    per_device_train_batch_size=2,       # Diturunkan ke 4 agar 1.5B parameter muat di T4
    gradient_accumulation_steps=4,       # Membantu batch size terasa seperti 8 tanpa membebani GPU
    save_steps=500,
    save_total_limit=2,
    fp16=True,                           # Akselerasi kecepatan training
    logging_steps=50,
    report_to="none"                     # Matikan notifikasi tracking eksternal
)

# 7. Inisialisasi SFTTrainer (Supervised Fine-Tuning Trainer)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
    # Argumen dataset_text_field dan max_seq_length sudah DIBUANG dari sini
)

print("\nMemulai proses Fine-Tuning Qwen2.5! Silakan tunggu... 🚀")
# Eksekusi Pelatihan
trainer.train()

# 8. Simpan Model Adaptor Final ke Folder Kaggle
trainer.save_model("/kaggle/working/qwen_fitness_lora_final")
tokenizer.save_pretrained("/kaggle/working/qwen_fitness_lora_final")

print("\nSelesai! Adaptor LoRA berhasil disimpan di /kaggle/working/qwen_fitness_lora_final")